# Cost-Sensitive Insurance Fraud Detection

Public adaptation of a CMU Data Science & Big Data team project by **Heedong Jang and Mengzhang Yin (Team T08)**.

Compare six classifiers, examine suspected identifier leakage, and select decision thresholds on training predictions before evaluating the held-out test set. The public adaptation fixes test-set threshold selection in the original cost analysis. Original saved outputs and claims of realized savings are not included.

Install `requirements.txt` from the repository root. Place an authorized copy of `carclaims.csv` in `data/`, or set `FRAUD_DATA_PATH`. Data is not distributed here. See the README for provenance and evaluation limits.

Cost assumptions are illustrative: false negative = $15,000; false positive = $500. This is a course benchmark, not a deployed insurer system.


### Part I: Exploratory Data Analysis


In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from thresholding import select_cost_threshold, total_cost
DATA_PATH_HJ = Path(os.environ.get("FRAUD_DATA_PATH", ROOT / "data" / "carclaims.csv"))
if not DATA_PATH_HJ.is_file():
    raise FileNotFoundError("Provide carclaims.csv in data/ or set FRAUD_DATA_PATH; see data/README.md")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings


sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FRAUD_PALETTE = {0: '#4C8BE0', 1: '#E05C4C'}

### 1.2 Load and Basic Overview


In [ ]:
df = pd.read_csv(DATA_PATH_HJ)
if not df['FraudFound'].isin(['Yes', 'No']).all():
    raise ValueError('FraudFound must contain only Yes/No; verify the dataset version.')
df['FraudFound'] = df['FraudFound'].map({'Yes': 1, 'No': 0})

print("=" * 55)
print(f"  Shape          : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"  Missing values : {df.isnull().sum().sum()}")
print(f"  Fraud cases    : {df['FraudFound'].sum():,}  ({df['FraudFound'].mean()*100:.1f}%)")
print(f"  Legit cases    : {(df['FraudFound']==0).sum():,}  ({(df['FraudFound']==0).mean()*100:.1f}%)")
print("=" * 55)
# Raw claim rows are intentionally not displayed.

### 1.3 Figure 1: Target Class Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['FraudFound'].value_counts().sort_index()
bars = ax.bar(['Legitimate', 'Fraud'], counts.values,
              color=['#4C8BE0', '#E05C4C'], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 150,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11)
ax.set_title('Target Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Claims')
ax.set_ylim(0, 17500)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 1.4 Figure 2: Numeric Feature Distributions by Fraud Label


In [ ]:
numeric_cols = ['Age', 'WeekOfMonth', 'WeekOfMonthClaimed', 'Deductible', 'DriverRating']

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(18, 4))
for ax, col in zip(axes, numeric_cols):
    for label, color in FRAUD_PALETTE.items():
        subset = df[df['FraudFound'] == label][col]
        ax.hist(subset, bins=15, alpha=0.6, color=color,
                label='Fraud' if label == 1 else 'Legitimate', density=True)
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('Density' if col == numeric_cols[0] else '')
axes[0].legend()
fig.suptitle('Numeric Feature Distributions by Fraud Label', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 1.5 Figure 3: Fraud Rate by Key Categorical Features


In [ ]:
cat_features = ['Fault', 'PoliceReportFiled', 'WitnessPresent', 'AccidentArea', 'AgentType', 'VehicleCategory']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, cat_features):
    fraud_rate = df.groupby(col)['FraudFound'].mean().sort_values(ascending=False)
    counts     = df.groupby(col)['FraudFound'].count()
    bars = ax.bar(fraud_rate.index, fraud_rate.values * 100,
                  color='#E05C4C', alpha=0.8, edgecolor='white')
    for bar, (cat, rate) in zip(bars, fraud_rate.items()):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.2,
                f'{rate*100:.1f}%\n(n={counts[cat]:,})',
                ha='center', va='bottom', fontsize=8.5)
    ax.set_title(f'Fraud Rate by {col}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_ylim(0, fraud_rate.max() * 100 * 1.45)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Fraud Rate by Key Categorical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 1.6 Figure 4: Fraud Rate by Ordinal Features


In [ ]:
ordinal_features = {
    'VehiclePrice': ['less than 20,000', '20,000 to 29,000', '30,000 to 39,000',
                     '40,000 to 59,000', '60,000 to 69,000', 'more than 69,000'],
    'PastNumberOfClaims': ['none', '1', '2 to 4', 'more than 4'],
    'AgeOfVehicle': ['new', '2 years', '3 years', '4 years',
                     '5 years', '6 years', '7 years', 'more than 7'],
    'NumberOfSuppliments': ['none', '1 to 2', '3 to 5', 'more than 5'],
    'AddressChange-Claim': ['no change', 'under 6 months', '1 year', '2 to 3 years', '4 to 8 years'],
}

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

for ax, (col, order) in zip(axes, ordinal_features.items()):
    order = [v for v in order if v in df[col].values]
    fraud_rate = df.groupby(col)['FraudFound'].mean().reindex(order)
    counts     = df.groupby(col)['FraudFound'].count().reindex(order)
    x = range(len(order))
    ax.bar(x, fraud_rate.values * 100, color='#4C8BE0', alpha=0.8, edgecolor='white')
    ax.plot(x, fraud_rate.values * 100, color='#1a3a6b', marker='o', linewidth=1.5)
    for xi, (rate, n) in enumerate(zip(fraud_rate.values, counts.values)):
        ax.text(xi, rate * 100 + 0.15, f'{rate*100:.1f}%', ha='center', fontsize=7.5)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=30, ha='right', fontsize=8)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_ylim(0, fraud_rate.max() * 100 * 1.4)

axes[-1].set_visible(False)
plt.suptitle('Fraud Rate Across Ordinal Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 1.7 Figure 5: Age Distribution, Fraud vs Legitimate


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, color, name in [(0, '#4C8BE0', 'Legitimate'), (1, '#E05C4C', 'Fraud')]:
    subset = df[df['FraudFound'] == label]['Age']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.axvline(subset.mean(), color=color, linestyle='--', linewidth=1.5,
               label=f'{name} mean: {subset.mean():.1f}')
ax.set_title('Claimant Age Distribution by Fraud Label', fontsize=13, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

### 1.8 Figure 6: Fraud Rate by Policy Type


In [ ]:
policy_fraud = (df.groupby('PolicyType')['FraudFound']
                  .agg(['mean', 'sum', 'count'])
                  .rename(columns={'mean': 'FraudRate', 'sum': 'FraudCount', 'count': 'Total'})
                  .sort_values('FraudRate', ascending=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors_p = ['#E05C4C' if r > policy_fraud['FraudRate'].mean() else '#4C8BE0'
            for r in policy_fraud['FraudRate']]
bars = ax.barh(policy_fraud.index, policy_fraud['FraudRate'] * 100,
               color=colors_p, edgecolor='white')
for bar, (idx, row) in zip(bars, policy_fraud.iterrows()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{row['FraudRate']*100:.1f}%  (n={int(row['Total']):,})",
            va='center', fontsize=9)
ax.axvline(policy_fraud['FraudRate'].mean() * 100, color='gray',
           linestyle='--', linewidth=1.2, label='Overall average')
ax.set_title('Fraud Rate by Policy Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Fraud Rate (%)')
ax.set_xlim(0, policy_fraud['FraudRate'].max() * 100 * 1.35)
ax.legend()
plt.tight_layout()
plt.show()

### 1.9 Figure 7: Fraud Rate by Month


In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_fraud = df.groupby('Month')['FraudFound'].mean().reindex(month_order)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(month_order, month_fraud.values * 100, color='#4C8BE0', alpha=0.8, edgecolor='white')
ax.plot(month_order, month_fraud.values * 100, color='#1a3a6b', marker='o', linewidth=1.5)
ax.axhline(df['FraudFound'].mean() * 100, color='tomato', linestyle='--',
           linewidth=1.2, label=f'Overall avg ({df["FraudFound"].mean()*100:.1f}%)')
ax.set_title('Fraud Rate by Month of Accident', fontsize=13, fontweight='bold')
ax.set_ylabel('Fraud Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

### 1.10 Correlation Heatmap


In [ ]:
ordinal_maps_eda = {
    'VehiclePrice':         {'less than 20,000':1,'20,000 to 29,000':2,'30,000 to 39,000':3,
                             '40,000 to 59,000':4,'60,000 to 69,000':5,'more than 69,000':6},
    'Days:Policy-Accident': {'none':0,'1 to 7':1,'8 to 15':2,'15 to 30':3,'more than 30':4},
    'Days:Policy-Claim':    {'none':0,'8 to 15':1,'15 to 30':2,'more than 30':3},
    'PastNumberOfClaims':   {'none':0,'1':1,'2 to 4':2,'more than 4':3},
    'AgeOfVehicle':         {'new':0,'2 years':1,'3 years':2,'4 years':3,
                             '5 years':4,'6 years':5,'7 years':6,'more than 7':7},
    'AgeOfPolicyHolder':    {'16 to 17':1,'18 to 20':2,'21 to 25':3,'26 to 30':4,
                             '31 to 35':5,'36 to 40':6,'41 to 50':7,'51 to 65':8,'over 65':9},
    'NumberOfSuppliments':  {'none':0,'1 to 2':1,'3 to 5':2,'more than 5':3},
    'AddressChange-Claim':  {'no change':0,'under 6 months':1,'1 year':2,
                             '2 to 3 years':3,'4 to 8 years':4},
    'NumberOfCars':         {'1 vehicle':1,'2 vehicles':2,'3 to 4':3,'5 to 8':4,'more than 8':5},
}
binary_maps_eda = {
    'Fault':             {'Policy Holder':0,'Third Party':1},
    'PoliceReportFiled': {'No':0,'Yes':1},
    'WitnessPresent':    {'No':0,'Yes':1},
    'AgentType':         {'Internal':0,'External':1},
    'AccidentArea':      {'Rural':0,'Urban':1},
    'Sex':               {'Male':0,'Female':1},
}

df_corr = df.copy()
for col, m in {**ordinal_maps_eda, **binary_maps_eda}.items():
    df_corr[col] = df_corr[col].map(m)

corr_cols = list(ordinal_maps_eda.keys()) + list(binary_maps_eda.keys()) + \
            ['Age', 'WeekOfMonth', 'Deductible', 'DriverRating', 'FraudFound']
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-0.4, vmax=0.4,
            linewidths=0.5, ax=ax, annot_kws={'size': 7.5})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.11 Summary Table: Top Fraud-Correlated Features


In [ ]:
fraud_corr = corr_matrix['FraudFound'].drop('FraudFound').sort_values(key=abs, ascending=False)
print("Top features correlated with FraudFound:")
print(fraud_corr.head(12).to_string())

### Part II: Modeling


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, precision_recall_curve,
                              precision_score, recall_score, f1_score, auc)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from matplotlib.patches import Patch

# Ordinal encoding
ordinal_maps = {
    'VehiclePrice':         {'less than 20,000':1,'20,000 to 29,000':2,'30,000 to 39,000':3,
                             '40,000 to 59,000':4,'60,000 to 69,000':5,'more than 69,000':6},
    'Days:Policy-Accident': {'none':0,'1 to 7':1,'8 to 15':2,'15 to 30':3,'more than 30':4},
    'Days:Policy-Claim':    {'none':0,'8 to 15':1,'15 to 30':2,'more than 30':3},
    'PastNumberOfClaims':   {'none':0,'1':1,'2 to 4':2,'more than 4':3},
    'AgeOfVehicle':         {'new':0,'2 years':1,'3 years':2,'4 years':3,
                             '5 years':4,'6 years':5,'7 years':6,'more than 7':7},
    'AgeOfPolicyHolder':    {'16 to 17':1,'18 to 20':2,'21 to 25':3,'26 to 30':4,
                             '31 to 35':5,'36 to 40':6,'41 to 50':7,'51 to 65':8,'over 65':9},
    'NumberOfSuppliments':  {'none':0,'1 to 2':1,'3 to 5':2,'more than 5':3},
    'AddressChange-Claim':  {'no change':0,'under 6 months':1,'1 year':2,
                             '2 to 3 years':3,'4 to 8 years':4},
    'NumberOfCars':         {'1 vehicle':1,'2 vehicles':2,'3 to 4':3,'5 to 8':4,'more than 8':5},
}
binary_maps = {
    'Fault':             {'Policy Holder':0,'Third Party':1},
    'PoliceReportFiled': {'No':0,'Yes':1},
    'WitnessPresent':    {'No':0,'Yes':1},
    'AgentType':         {'Internal':0,'External':1},
    'AccidentArea':      {'Rural':0,'Urban':1},
    'Sex':               {'Male':0,'Female':1},
}

# HJ: dropped ID/temporal leak columns edited
# PolicyNumber is a sequential ID (1..15420) that perfectly encodes
# temporal ordering of the dataset. RepNumber and Year are also
# administrative fields that should not drive a fraud decision in
# production. Dropping them prevents the models from picking up
# spurious temporal signal.
df_model = df.drop(columns=['PolicyNumber', 'RepNumber', 'Year']).copy()
for col, m in {**ordinal_maps, **binary_maps}.items():
    df_model[col] = df_model[col].map(m)

# Interaction feature: policyholder at fault (Fault=0) + external agent (AgentType=1)
# This combination showed the highest fraud rate in EDA and is encoded as an explicit binary signal
df_model['Fault_x_ExternalAgent'] = (
    (df_model['Fault'] == 0) & (df_model['AgentType'] == 1)
).astype(int)

# One-hot encode remaining categorical columns
cat_cols = df_model.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

# Stratified train/test split preserves the 6% fraud rate in both sets
X = df_model.drop('FraudFound', axis=1)
y = df_model['FraudFound']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape[0]:,}  |  Fraud in train : {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Test size  : {X_test.shape[0]:,}  |  Fraud in test  : {y_test.sum():,} ({y_test.mean()*100:.1f}%)")
print(f"Features   : {X_train.shape[1]}")

# HJ: evaluation helper edited (style guide: functional abstraction)
def fraud_metrics(y_true, y_pred, y_prob):
    '''
    Compute the four headline metrics for the fraud (positive) class.

    Returns a dict with precision, recall, f1, and PR-AUC.
    Used throughout the modeling section so each model is scored
    the same way without copy-pasting the metric calls.
    '''
    p, r, _ = precision_recall_curve(y_true, y_prob)
    return {
        'precision': precision_score(y_true, y_pred, pos_label=1),
        'recall':    recall_score(y_true, y_pred, pos_label=1),
        'f1':        f1_score(y_true, y_pred, pos_label=1),
        'auc_pr':    auc(r, p),
    }


### 2.2 Logistic Regression


In [ ]:
# 2.2.1 Baseline Logistic Regression
lr_base = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])
lr_base.fit(X_train, y_train)
y_pred_lr_base = lr_base.predict(X_test)
print("Baseline Logistic Regression")
print(classification_report(y_test, y_pred_lr_base, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.2.2 Hyperparameter Tuning
# HJ: GridSearch restored edited (was hardcoded with class_weight={0:1, 1:8})
param_grid_lr = {
    'lr__C':            [0.1, 1, 10],
    'lr__penalty':      ['l2'],
    'lr__class_weight': [{0: 1, 1: 5}, {0: 1, 1: 8}, 'balanced'],
    'lr__solver':       ['liblinear'],
}
gs_lr = GridSearchCV(
    Pipeline([('scaler', StandardScaler()),
              ('lr', LogisticRegression(random_state=42,
                                        max_iter=500))]),
    param_grid_lr, scoring='f1', cv=3, n_jobs=2, verbose=0
)
gs_lr.fit(X_train, y_train)
best_lr = gs_lr.best_estimator_
print(f"Best params : {gs_lr.best_params_}")
print(f"Best CV F1  : {gs_lr.best_score_:.4f}")

In [ ]:
# 2.2.3 Evaluate Tuned Model at Default Threshold
y_pred_lr = best_lr.predict(X_test)
y_prob_lr = best_lr.predict_proba(X_test)[:, 1]
pr_auc_lr = auc(*precision_recall_curve(y_test, y_prob_lr)[:2][::-1])

print("Tuned Logistic Regression (default threshold 0.5)")
print(classification_report(y_test, y_pred_lr, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.2.4 Threshold Tuning via Out-of-Fold Predictions on Training Set
y_prob_lr_oof = cross_val_predict(best_lr, X_train, y_train, cv=5, method='predict_proba')[:, 1]
prec_oof, rec_oof, thresh_oof = precision_recall_curve(y_train, y_prob_lr_oof)
f1_oof = 2 * prec_oof * rec_oof / (prec_oof + rec_oof + 1e-8)
best_thresh_lr = thresh_oof[np.argmax(f1_oof[:-1])]

y_pred_lr_tuned = (y_prob_lr >= best_thresh_lr).astype(int)

precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_prob_lr)
best_idx_lr = np.argmax(2 * precision_lr * recall_lr / (precision_lr + recall_lr + 1e-8))

print(f"Best threshold (OOF) : {best_thresh_lr:.3f}")
print(f"Precision            : {precision_score(y_test, y_pred_lr_tuned, pos_label=1):.3f}")
print(f"Recall               : {recall_score(y_test, y_pred_lr_tuned, pos_label=1):.3f}")
print(f"F1                   : {f1_score(y_test, y_pred_lr_tuned, pos_label=1):.3f}")
print(f"AUC-PR               : {pr_auc_lr:.3f}")
print()
print("After Threshold Tuning")
print(classification_report(y_test, y_pred_lr_tuned, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.2.5 Precision-Recall Curve
plt.figure(figsize=(7, 5))
plt.plot(recall_lr, precision_lr, color='steelblue', lw=2, label=f'AUC-PR = {pr_auc_lr:.3f}')
plt.axvline(recall_score(y_test, y_pred_lr_tuned, zero_division=0), color='tomato', linestyle='--',
            label=f'Threshold = {best_thresh_lr:.2f}')
plt.scatter(recall_score(y_test, y_pred_lr_tuned, zero_division=0), precision_score(y_test, y_pred_lr_tuned, zero_division=0), color='tomato', s=80, zorder=5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Logistic Regression', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.2.6 Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr_tuned)
ConfusionMatrixDisplay(cm_lr, display_labels=['Legitimate', 'Fraud']).plot(colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 2.2.7 Top 15 Feature Coefficients
coefs = pd.Series(best_lr.named_steps['lr'].coef_[0], index=X_train.columns)
top15_lr = coefs.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
colors_lr = ['#E05C4C' if coefs[f] > 0 else '#4C8BE0' for f in top15_lr.index]
ax.barh(top15_lr.index[::-1], top15_lr.values[::-1], color=colors_lr[::-1], alpha=0.85)
ax.set_title('Top 15 Features by Absolute Coefficient — Logistic Regression', fontweight='bold')
ax.set_xlabel('Absolute Coefficient')
ax.grid(alpha=0.3, axis='x')
ax.legend(handles=[Patch(color='#E05C4C', label='Increases fraud probability'),
                   Patch(color='#4C8BE0', label='Decreases fraud probability')],
          loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 2.2.8 Logistic Regression Results Summary
summary_lr = pd.DataFrame({
    'Model':     ['LR Baseline', 'LR Tuned (threshold 0.5)', 'LR Tuned (OOF threshold)'],
    'Precision': [precision_score(y_test, y_pred_lr_base,  pos_label=1),
                  precision_score(y_test, y_pred_lr,        pos_label=1),
                  precision_score(y_test, y_pred_lr_tuned,  pos_label=1)],
    'Recall':    [recall_score(y_test, y_pred_lr_base,  pos_label=1),
                  recall_score(y_test, y_pred_lr,        pos_label=1),
                  recall_score(y_test, y_pred_lr_tuned,  pos_label=1)],
    'F1':        [f1_score(y_test, y_pred_lr_base,  pos_label=1),
                  f1_score(y_test, y_pred_lr,        pos_label=1),
                  f1_score(y_test, y_pred_lr_tuned,  pos_label=1)],
    'AUC-PR':   [auc(*precision_recall_curve(y_test, lr_base.predict_proba(X_test)[:,1])[:2][::-1]),
                 pr_auc_lr, pr_auc_lr],
})
print("Logistic Regression Results Summary")
print(summary_lr.round(4).to_string(index=False))

### 2.3 Decision Tree


In [ ]:
# 2.3.1 Baseline Decision Tree
dt_base = Pipeline([
    ('dt', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])
dt_base.fit(X_train, y_train)
y_pred_dt_base = dt_base.predict(X_test)
print("Baseline Decision Tree")
print(classification_report(y_test, y_pred_dt_base, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.3.2 Hyperparameter Tuning with class_weight as a tunable parameter
param_grid_dt = {
    'dt__max_depth':        [3, 5, 7, 10],
    'dt__min_samples_leaf': [5, 10, 20],
    'dt__criterion':        ['gini', 'entropy'],
    'dt__class_weight':     [{0:1, 1:5}, {0:1, 1:8}, 'balanced'],
}
gs_dt = GridSearchCV(
    Pipeline([('dt', DecisionTreeClassifier(random_state=42))]),
    param_grid_dt, scoring='f1', cv=3, n_jobs=2, verbose=0
)
gs_dt.fit(X_train, y_train)
print(f"Best params : {gs_dt.best_params_}")
print(f"Best CV F1  : {gs_dt.best_score_:.4f}")

In [ ]:
# 2.3.3 Evaluate Tuned Model at Default Threshold
best_dt   = gs_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)
y_prob_dt = best_dt.predict_proba(X_test)[:, 1]
pr_auc_dt = auc(*precision_recall_curve(y_test, y_prob_dt)[:2][::-1])

print("Tuned Decision Tree (default threshold 0.5)")
print(classification_report(y_test, y_pred_dt, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.3.4 Threshold Tuning via Out-of-Fold Predictions on Training Set
y_prob_dt_oof = cross_val_predict(best_dt, X_train, y_train, cv=5, method='predict_proba')[:, 1]
prec_oof, rec_oof, thresh_oof = precision_recall_curve(y_train, y_prob_dt_oof)
f1_oof = 2 * prec_oof * rec_oof / (prec_oof + rec_oof + 1e-8)
best_thresh_dt = thresh_oof[np.argmax(f1_oof[:-1])]

y_pred_dt_tuned = (y_prob_dt >= best_thresh_dt).astype(int)

precision_dt, recall_dt, _ = precision_recall_curve(y_test, y_prob_dt)
best_idx_dt = np.argmax(2 * precision_dt * recall_dt / (precision_dt + recall_dt + 1e-8))

print(f"Best threshold (OOF) : {best_thresh_dt:.3f}")
print(f"Precision            : {precision_score(y_test, y_pred_dt_tuned, pos_label=1):.3f}")
print(f"Recall               : {recall_score(y_test, y_pred_dt_tuned, pos_label=1):.3f}")
print(f"F1                   : {f1_score(y_test, y_pred_dt_tuned, pos_label=1):.3f}")
print(f"AUC-PR               : {pr_auc_dt:.4f}")
print()
print("After Threshold Tuning")
print(classification_report(y_test, y_pred_dt_tuned, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.3.5 Precision-Recall Curve
plt.figure(figsize=(7, 5))
plt.plot(recall_dt, precision_dt, color='steelblue', lw=2, label=f'AUC-PR = {pr_auc_dt:.3f}')
plt.axvline(recall_score(y_test, y_pred_dt_tuned, zero_division=0), color='tomato', linestyle='--',
            label=f'Threshold = {best_thresh_dt:.2f}')
plt.scatter(recall_score(y_test, y_pred_dt_tuned, zero_division=0), precision_score(y_test, y_pred_dt_tuned, zero_division=0), color='tomato', s=80, zorder=5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Decision Tree', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.3.6 Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt_tuned)
ConfusionMatrixDisplay(cm_dt, display_labels=['Legitimate', 'Fraud']).plot(colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — Decision Tree', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 2.3.7 Top 15 Feature Importances
importances_dt = pd.Series(best_dt.named_steps['dt'].feature_importances_, index=X_train.columns)
top15_dt = importances_dt.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
plt.barh(top15_dt.index[::-1], top15_dt.values[::-1], color='steelblue', alpha=0.85)
plt.title('Top 15 Feature Importances — Decision Tree', fontweight='bold')
plt.xlabel('Importance Score'); plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# 2.3.8 Decision Tree Results Summary
summary_dt = pd.DataFrame({
    'Model':     ['DT Baseline', 'DT Tuned (threshold 0.5)', 'DT Tuned (OOF threshold)'],
    'Precision': [precision_score(y_test, y_pred_dt_base,  pos_label=1),
                  precision_score(y_test, y_pred_dt,        pos_label=1),
                  precision_score(y_test, y_pred_dt_tuned,  pos_label=1)],
    'Recall':    [recall_score(y_test, y_pred_dt_base,  pos_label=1),
                  recall_score(y_test, y_pred_dt,        pos_label=1),
                  recall_score(y_test, y_pred_dt_tuned,  pos_label=1)],
    'F1':        [f1_score(y_test, y_pred_dt_base,  pos_label=1),
                  f1_score(y_test, y_pred_dt,        pos_label=1),
                  f1_score(y_test, y_pred_dt_tuned,  pos_label=1)],
    'AUC-PR':   [auc(*precision_recall_curve(y_test, dt_base.predict_proba(X_test)[:,1])[:2][::-1]),
                 pr_auc_dt, pr_auc_dt],
})
print("Decision Tree Results Summary")
print(summary_dt.round(4).to_string(index=False))

### 2.4 Random Forest


In [ ]:
# 2.4.1 Baseline Random Forest
rf_base = Pipeline([
    ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                   random_state=42, n_jobs=2))
])
rf_base.fit(X_train, y_train)
y_pred_rf_base = rf_base.predict(X_test)
print("Baseline Random Forest (200 trees)")
print(classification_report(y_test, y_pred_rf_base, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.4.2 Hyperparameter Tuning — reduced grid
param_grid_rf = {
    'rf__n_estimators':     [100, 200],
    # HJ: removed max_depth=None edited to limit RF overfitting
    'rf__max_depth':        [5, 10, 15],
    'rf__min_samples_leaf': [5, 10],
    'rf__class_weight':     [{0:1, 1:5}, {0:1, 1:8}, 'balanced'],
}
gs_rf = GridSearchCV(
    Pipeline([('rf', RandomForestClassifier(random_state=42, n_jobs=2))]),
    param_grid_rf, scoring='f1', cv=3, n_jobs=2, verbose=1
)
gs_rf.fit(X_train, y_train)
print(f"Best params : {gs_rf.best_params_}")
print(f"Best CV F1  : {gs_rf.best_score_:.4f}")

In [ ]:
# 2.4.3 Evaluate Tuned Model at Default Threshold
best_rf   = gs_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]
pr_auc_rf = auc(*precision_recall_curve(y_test, y_prob_rf)[:2][::-1])

print("Tuned Random Forest (default threshold 0.5)")
print(classification_report(y_test, y_pred_rf, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.4.4 Threshold Tuning via Out-of-Fold Predictions on Training Set
y_prob_rf_oof = cross_val_predict(best_rf, X_train, y_train, cv=5, method='predict_proba')[:, 1]
prec_oof, rec_oof, thresh_oof = precision_recall_curve(y_train, y_prob_rf_oof)
f1_oof = 2 * prec_oof * rec_oof / (prec_oof + rec_oof + 1e-8)
best_thresh_rf = thresh_oof[np.argmax(f1_oof[:-1])]

y_pred_rf_tuned = (y_prob_rf >= best_thresh_rf).astype(int)

precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_prob_rf)
best_idx_rf = np.argmax(2 * precision_rf * recall_rf / (precision_rf + recall_rf + 1e-8))

print(f"Best threshold (OOF) : {best_thresh_rf:.3f}")
print(f"Precision            : {precision_score(y_test, y_pred_rf_tuned, pos_label=1):.3f}")
print(f"Recall               : {recall_score(y_test, y_pred_rf_tuned, pos_label=1):.3f}")
print(f"F1                   : {f1_score(y_test, y_pred_rf_tuned, pos_label=1):.3f}")
print(f"AUC-PR               : {pr_auc_rf:.3f}")
print()
print("After Threshold Tuning")
print(classification_report(y_test, y_pred_rf_tuned, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.4.5 Precision-Recall Curve
plt.figure(figsize=(7, 5))
plt.plot(recall_rf, precision_rf, color='steelblue', lw=2, label=f'AUC-PR = {pr_auc_rf:.3f}')
plt.axvline(recall_score(y_test, y_pred_rf_tuned, zero_division=0), color='tomato', linestyle='--',
            label=f'Threshold = {best_thresh_rf:.2f}')
plt.scatter(recall_score(y_test, y_pred_rf_tuned, zero_division=0), precision_score(y_test, y_pred_rf_tuned, zero_division=0), color='tomato', s=80, zorder=5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Random Forest', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.4.6 Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf_tuned)
ConfusionMatrixDisplay(cm_rf, display_labels=['Legitimate', 'Fraud']).plot(colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — Random Forest', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 2.4.7 Top 15 Feature Importances
importances_rf = pd.Series(best_rf.named_steps['rf'].feature_importances_, index=X_train.columns)
top15_rf = importances_rf.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
plt.barh(top15_rf.index[::-1], top15_rf.values[::-1], color='steelblue', alpha=0.85)
plt.title('Top 15 Feature Importances — Random Forest', fontweight='bold')
plt.xlabel('Importance Score'); plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# 2.4.8 Random Forest Results Summary
summary_rf = pd.DataFrame({
    'Model':     ['RF Baseline', 'RF Tuned (threshold 0.5)', 'RF Tuned (OOF threshold)'],
    'Precision': [precision_score(y_test, y_pred_rf_base,  pos_label=1),
                  precision_score(y_test, y_pred_rf,        pos_label=1),
                  precision_score(y_test, y_pred_rf_tuned,  pos_label=1)],
    'Recall':    [recall_score(y_test, y_pred_rf_base,  pos_label=1),
                  recall_score(y_test, y_pred_rf,        pos_label=1),
                  recall_score(y_test, y_pred_rf_tuned,  pos_label=1)],
    'F1':        [f1_score(y_test, y_pred_rf_base,  pos_label=1),
                  f1_score(y_test, y_pred_rf,        pos_label=1),
                  f1_score(y_test, y_pred_rf_tuned,  pos_label=1)],
    'AUC-PR':   [auc(*precision_recall_curve(y_test, rf_base.predict_proba(X_test)[:,1])[:2][::-1]),
                 pr_auc_rf, pr_auc_rf],
})
print("Random Forest Results Summary")
print(summary_rf.round(4).to_string(index=False))

### 2.5 XGBoost


In [ ]:
# HJ: pip install moved to top edited
# 2.5.1 Install and Import XGBoost
from xgboost import XGBClassifier

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Full class ratio (scale_pos_weight baseline) = {SCALE_POS_WEIGHT:.2f}")

In [ ]:
# 2.5.2 Baseline XGBoost
xgb_base = Pipeline([
    ('xgb', XGBClassifier(
        n_estimators     = 200,
        scale_pos_weight = SCALE_POS_WEIGHT,
        random_state     = 42,
        eval_metric      = 'aucpr',
        n_jobs           = -1,
        verbosity        = 0
    ))
])
xgb_base.fit(X_train, y_train)
y_pred_xgb_base = xgb_base.predict(X_test)
print("Baseline XGBoost")
print(classification_report(y_test, y_pred_xgb_base, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.5.3 Hyperparameter Tuning — reduced grid
param_grid_xgb = {
    'xgb__n_estimators':     [100, 200],
    'xgb__max_depth':        [3, 6],
    'xgb__learning_rate':    [0.05, 0.1],
    'xgb__subsample':        [0.8],
    'xgb__colsample_bytree': [0.8],
    'xgb__scale_pos_weight': [5, 8, round(SCALE_POS_WEIGHT)],
}
gs_xgb = GridSearchCV(
    Pipeline([
        ('xgb', XGBClassifier(random_state=42, eval_metric='aucpr', n_jobs=2, verbosity=0))
    ]),
    param_grid_xgb, scoring='f1', cv=3, n_jobs=2, verbose=1
)
gs_xgb.fit(X_train, y_train)
print(f"Best params : {gs_xgb.best_params_}")
print(f"Best CV F1  : {gs_xgb.best_score_:.4f}")

In [ ]:
# 2.5.4 Evaluate Tuned Model at Default Threshold
best_xgb   = gs_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]
pr_auc_xgb = auc(*precision_recall_curve(y_test, y_prob_xgb)[:2][::-1])

print("Tuned XGBoost (default threshold 0.5)")
print(classification_report(y_test, y_pred_xgb, target_names=['Legitimate', 'Fraud']))

In [ ]:
# 2.5.5 Threshold Tuning via Out-of-Fold Predictions on Training Set
y_prob_xgb_oof = cross_val_predict(best_xgb, X_train, y_train, cv=5, method='predict_proba')[:, 1]
prec_oof, rec_oof, thresh_oof = precision_recall_curve(y_train, y_prob_xgb_oof)
f1_oof = 2 * prec_oof * rec_oof / (prec_oof + rec_oof + 1e-8)
best_thresh_xgb = thresh_oof[np.argmax(f1_oof[:-1])]

y_pred_xgb_tuned = (y_prob_xgb >= best_thresh_xgb).astype(int)

precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_prob_xgb)
best_idx_xgb = np.argmax(2 * precision_xgb * recall_xgb / (precision_xgb + recall_xgb + 1e-8))

print(f"Best threshold (OOF) : {best_thresh_xgb:.3f}")
print(f"Precision            : {precision_score(y_test, y_pred_xgb_tuned, pos_label=1):.3f}")
print(f"Recall               : {recall_score(y_test, y_pred_xgb_tuned, pos_label=1):.3f}")
print(f"F1                   : {f1_score(y_test, y_pred_xgb_tuned, pos_label=1):.3f}")
print(f"AUC-PR               : {pr_auc_xgb:.4f}")
print()
print("After Threshold Tuning")
print(classification_report(y_test, y_pred_xgb_tuned, target_names=['Legitimate', 'Fraud']))

### 2.5.5b Cost-Based Threshold for XGBoost


In [ ]:
# Select the cost threshold only on out-of-fold TRAINING predictions.
C_MISS, C_FP = 15000, 500
grid = np.linspace(0.05, 0.95, 91)
cost_thresh_xgb, costs = select_cost_threshold(y_train, y_prob_xgb_oof, grid, C_MISS, C_FP)
y_pred_xgb_cost = (y_prob_xgb >= cost_thresh_xgb).astype(int)
m_cost = fraud_metrics(y_test, y_pred_xgb_cost, y_prob_xgb)
f1_test_cost = total_cost(best_thresh_xgb, y_test, y_prob_xgb, C_MISS, C_FP)
cost_test_cost = total_cost(cost_thresh_xgb, y_test, y_prob_xgb, C_MISS, C_FP)
print(f"OOF F1 threshold: {best_thresh_xgb:.3f}; OOF cost threshold: {cost_thresh_xgb:.3f}")
print(f"Held-out modeled cost at F1 threshold: ${f1_test_cost:,.0f}")
print(f"Held-out modeled cost at cost threshold: ${cost_test_cost:,.0f}")
print(f"Difference under stated assumptions: ${f1_test_cost - cost_test_cost:,.0f}")
print("Held-out cost-threshold metrics:", m_cost)
plt.figure(figsize=(7, 4))
plt.plot(grid, costs / 1000, color='steelblue', lw=2)
plt.axvline(cost_thresh_xgb, color='tomato', linestyle='--', label=f'OOF cost threshold: {cost_thresh_xgb:.2f}')
plt.axvline(best_thresh_xgb, color='gray', linestyle=':', label=f'OOF F1 threshold: {best_thresh_xgb:.2f}')
plt.xlabel('Decision threshold'); plt.ylabel('Training OOF modeled cost ($K)')
plt.title('Training-only threshold selection')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 2.5.6 Precision-Recall Curve
plt.figure(figsize=(7, 5))
plt.plot(recall_xgb, precision_xgb, color='steelblue', lw=2, label=f'AUC-PR = {pr_auc_xgb:.3f}')
plt.axvline(recall_score(y_test, y_pred_xgb_tuned, zero_division=0), color='tomato', linestyle='--',
            label=f'Threshold = {best_thresh_xgb:.2f}')
plt.scatter(recall_score(y_test, y_pred_xgb_tuned, zero_division=0), precision_score(y_test, y_pred_xgb_tuned, zero_division=0), color='tomato', s=80, zorder=5)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve — XGBoost', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.5.7 Confusion Matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb_tuned)
ConfusionMatrixDisplay(cm_xgb, display_labels=['Legitimate', 'Fraud']).plot(colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — XGBoost', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 2.5.8 Top 15 Feature Importances
importances_xgb = pd.Series(
    best_xgb.named_steps['xgb'].feature_importances_,
    index=X_train.columns
)
top15_xgb = importances_xgb.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
plt.barh(top15_xgb.index[::-1], top15_xgb.values[::-1], color='steelblue', alpha=0.85)
plt.title('Top 15 Feature Importances — XGBoost', fontweight='bold')
plt.xlabel('Importance Score'); plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# 2.5.9 XGBoost Results Summary
summary_xgb = pd.DataFrame({
    'Model':     ['XGB Baseline', 'XGB Tuned (threshold 0.5)', 'XGB Tuned (OOF threshold)'],
    'Precision': [precision_score(y_test, y_pred_xgb_base,  pos_label=1),
                  precision_score(y_test, y_pred_xgb,        pos_label=1),
                  precision_score(y_test, y_pred_xgb_tuned,  pos_label=1)],
    'Recall':    [recall_score(y_test, y_pred_xgb_base,  pos_label=1),
                  recall_score(y_test, y_pred_xgb,        pos_label=1),
                  recall_score(y_test, y_pred_xgb_tuned,  pos_label=1)],
    'F1':        [f1_score(y_test, y_pred_xgb_base,  pos_label=1),
                  f1_score(y_test, y_pred_xgb,        pos_label=1),
                  f1_score(y_test, y_pred_xgb_tuned,  pos_label=1)],
    'AUC-PR':   [auc(*precision_recall_curve(y_test, xgb_base.predict_proba(X_test)[:,1])[:2][::-1]),
                 pr_auc_xgb, pr_auc_xgb],
})
print("XGBoost Results Summary")
print(summary_xgb.round(4).to_string(index=False))

### 2.6 CatBoost


In [ ]:
# 2.6 CatBoost
# HJ: CatBoost added edited (native categorical handling, public-benchmark comparison)
from catboost import CatBoostClassifier

# Re-prepare data without one-hot encoding for CatBoost
df_cb = df.drop(columns=['PolicyNumber', 'RepNumber', 'Year']).copy()
df_cb['Fault_x_ExternalAgent'] = (
    (df_cb['Fault'] == 'Policy Holder') & (df_cb['AgentType'] == 'External')
).astype(int)
X_cb = df_cb.drop('FraudFound', axis=1)
y_cb = df_cb['FraudFound']

cat_features = X_cb.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
print(f"CatBoost categorical features: {len(cat_features)}")

# Outer split: train+val vs test (matches other models)
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_cb, y_cb, test_size=0.2, random_state=42, stratify=y_cb
)

# Inner split for threshold tuning: train_inner (80%) vs val (20%)
X_tr_inner, X_val_cb, y_tr_inner, y_val_cb = train_test_split(
    X_train_cb, y_train_cb, test_size=0.2, random_state=42, stratify=y_train_cb
)

cb_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    cat_features=cat_features,
    class_weights={0: 1, 1: 8},
    eval_metric='PRAUC',
    random_state=42,
    verbose=False,
    thread_count=2,
    allow_writing_files=False,
)
cb_model.fit(X_tr_inner, y_tr_inner)

# Validation-set threshold tuning (analogue to OOF for tree-native categorical models)
y_prob_val_cb = cb_model.predict_proba(X_val_cb)[:, 1]
p_v, r_v, t_v = precision_recall_curve(y_val_cb, y_prob_val_cb)
f1_v = 2 * p_v * r_v / (p_v + r_v + 1e-8)
best_thresh_cb = t_v[np.argmax(f1_v[:-1])]

# Refit on full training data for final evaluation
cb_model.fit(X_train_cb, y_train_cb)
y_prob_cb = cb_model.predict_proba(X_test_cb)[:, 1]
prec_cb, rec_cb, _ = precision_recall_curve(y_test_cb, y_prob_cb)
pr_auc_cb = auc(rec_cb, prec_cb)
y_pred_cb_tuned = (y_prob_cb >= best_thresh_cb).astype(int)

print(f"CatBoost val-set threshold : {best_thresh_cb:.3f}")
print(f"Precision : {precision_score(y_test_cb, y_pred_cb_tuned, pos_label=1):.3f}")
print(f"Recall    : {recall_score(y_test_cb, y_pred_cb_tuned, pos_label=1):.3f}")
print(f"F1        : {f1_score(y_test_cb, y_pred_cb_tuned, pos_label=1):.3f}")
print(f"AUC-PR    : {pr_auc_cb:.4f}")
print()
print(classification_report(y_test_cb, y_pred_cb_tuned, target_names=['Legitimate', 'Fraud']))

### 2.7 LightGBM


In [ ]:
# 2.7 LightGBM
# HJ: LightGBM added edited (gradient boosting with native categorical handling)
import lightgbm as lgb
import re

# Sanitize column names: LightGBM rejects ':', '-', spaces in feature names
def clean_col(c):
    return re.sub(r'[^A-Za-z0-9_]', '_', str(c))


X_train_lgb = X_train.rename(columns=clean_col)
X_test_lgb  = X_test.rename(columns=clean_col)

SCALE_POS_WEIGHT_LGB = (y_train == 0).sum() / (y_train == 1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=SCALE_POS_WEIGHT_LGB,
    random_state=42,
    verbose=-1,
    n_jobs=2,
)
lgb_model.fit(X_train_lgb, y_train)

y_prob_lgb = lgb_model.predict_proba(X_test_lgb)[:, 1]
prec_lgb, rec_lgb, _ = precision_recall_curve(y_test, y_prob_lgb)
pr_auc_lgb = auc(rec_lgb, prec_lgb)

# OOF threshold tuning
y_prob_lgb_oof = cross_val_predict(lgb_model, X_train_lgb, y_train,
                                    cv=3, method='predict_proba')[:, 1]
p_oof, r_oof, t_oof = precision_recall_curve(y_train, y_prob_lgb_oof)
f1_oof = 2 * p_oof * r_oof / (p_oof + r_oof + 1e-8)
best_thresh_lgb = t_oof[np.argmax(f1_oof[:-1])]
y_pred_lgb_tuned = (y_prob_lgb >= best_thresh_lgb).astype(int)

print(f"LightGBM OOF threshold : {best_thresh_lgb:.3f}")
print(f"Precision : {precision_score(y_test, y_pred_lgb_tuned, pos_label=1):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred_lgb_tuned, pos_label=1):.3f}")
print(f"F1        : {f1_score(y_test, y_pred_lgb_tuned, pos_label=1):.3f}")
print(f"AUC-PR    : {pr_auc_lgb:.4f}")
print()
print(classification_report(y_test, y_pred_lgb_tuned, target_names=['Legitimate', 'Fraud']))

### 2.8 Diagnostic: Unweighted vs Class-Weighted Models


In [ ]:
# 2.8 Unweighted vs Class-Weighted Diagnostic
from xgboost import XGBClassifier

# Logistic Regression (no class weight)
lr_uw = Pipeline([('scaler', StandardScaler()),
                  ('lr', LogisticRegression(C=1, penalty='l2', solver='liblinear',
                                             random_state=42, max_iter=500))])
lr_uw.fit(X_train, y_train)
y_prob_lr_uw  = lr_uw.predict_proba(X_test)[:, 1]
pr_auc_lr_uw  = auc(*precision_recall_curve(y_test, y_prob_lr_uw)[:2][::-1])
y_pred_lr_uw  = (y_prob_lr_uw >= 0.5).astype(int)

# Decision Tree (no class weight)
dt_uw = Pipeline([('dt', DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42))])
dt_uw.fit(X_train, y_train)
y_prob_dt_uw  = dt_uw.predict_proba(X_test)[:, 1]
pr_auc_dt_uw  = auc(*precision_recall_curve(y_test, y_prob_dt_uw)[:2][::-1])
y_pred_dt_uw  = (y_prob_dt_uw >= 0.5).astype(int)

# Random Forest (no class weight)
rf_uw = Pipeline([('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=2))])
rf_uw.fit(X_train, y_train)
y_prob_rf_uw  = rf_uw.predict_proba(X_test)[:, 1]
pr_auc_rf_uw  = auc(*precision_recall_curve(y_test, y_prob_rf_uw)[:2][::-1])
y_pred_rf_uw  = (y_prob_rf_uw >= 0.5).astype(int)

# XGBoost (scale_pos_weight=1, i.e., no imbalance adjustment)
xgb_uw = Pipeline([('xgb', XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                                           scale_pos_weight=1, random_state=42,
                                           eval_metric='aucpr', n_jobs=2, verbosity=0))])
xgb_uw.fit(X_train, y_train)
y_prob_xgb_uw = xgb_uw.predict_proba(X_test)[:, 1]
pr_auc_xgb_uw = auc(*precision_recall_curve(y_test, y_prob_xgb_uw)[:2][::-1])
y_pred_xgb_uw = (y_prob_xgb_uw >= 0.5).astype(int)

# Comparison table
diag_df = pd.DataFrame({
    'Model':               ['LR', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'F1 (unweighted)':     [f1_score(y_test, y_pred_lr_uw,    pos_label=1),
                            f1_score(y_test, y_pred_dt_uw,    pos_label=1),
                            f1_score(y_test, y_pred_rf_uw,    pos_label=1),
                            f1_score(y_test, y_pred_xgb_uw,   pos_label=1)],
    'F1 (weighted)':       [f1_score(y_test, y_pred_lr_tuned,  pos_label=1),
                            f1_score(y_test, y_pred_dt_tuned,  pos_label=1),
                            f1_score(y_test, y_pred_rf_tuned,  pos_label=1),
                            f1_score(y_test, y_pred_xgb_tuned, pos_label=1)],
    'AUC-PR (unweighted)': [pr_auc_lr_uw, pr_auc_dt_uw, pr_auc_rf_uw, pr_auc_xgb_uw],
    'AUC-PR (weighted)':   [pr_auc_lr,    pr_auc_dt,    pr_auc_rf,    pr_auc_xgb],
})
diag_df['F1 lift']     = (diag_df['F1 (weighted)']     - diag_df['F1 (unweighted)']).round(4)
diag_df['AUC-PR lift'] = (diag_df['AUC-PR (weighted)'] - diag_df['AUC-PR (unweighted)']).round(4)
print("Unweighted vs Class-Weighted Diagnostic")
print(diag_df.round(4).to_string(index=False))

# Visual
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Unweighted vs Class-Weighted: F1 and AUC-PR by Model',
             fontsize=13, fontweight='bold')
models_d = diag_df['Model'].tolist()
xd = np.arange(len(models_d)); wd = 0.35
for ax, col_uw, col_w, title, ylabel in [
    (axes[0], 'F1 (unweighted)',     'F1 (weighted)',     'F1 (Fraud Class)', 'F1'),
    (axes[1], 'AUC-PR (unweighted)', 'AUC-PR (weighted)', 'AUC-PR',           'AUC-PR'),
]:
    b1 = ax.bar(xd - wd/2, diag_df[col_uw], wd, label='Unweighted',       color='#B0BEC5', alpha=0.9, edgecolor='white')
    b2 = ax.bar(xd + wd/2, diag_df[col_w],  wd, label='Weighted (tuned)', color='#4C8BE0', alpha=0.9, edgecolor='white')
    ax.set_xticks(xd); ax.set_xticklabels(models_d, fontsize=10)
    ax.set_ylim(0, max(diag_df[col_w].max(), diag_df[col_uw].max()) * 1.3)
    ax.set_title(title, fontweight='bold'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    for bar in b1:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=8, color='#555')
    for bar in b2:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=8, color='#1a3a6b', fontweight='bold')
plt.tight_layout(); plt.show()


## Part III: Final Evaluation


In [ ]:
# Final results table with all 6 models, both AUC-PR and AUC-ROC
# HJ: extended to 6 models + AUC-ROC for benchmark comparison edited
from sklearn.metrics import roc_auc_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest',
              'XGBoost', 'CatBoost', 'LightGBM'],
    'Precision': [precision_score(y_test, y_pred_lr_tuned,  pos_label=1),
                  precision_score(y_test, y_pred_dt_tuned,  pos_label=1),
                  precision_score(y_test, y_pred_rf_tuned,  pos_label=1),
                  precision_score(y_test, y_pred_xgb_tuned, pos_label=1),
                  precision_score(y_test_cb, y_pred_cb_tuned, pos_label=1),
                  precision_score(y_test, y_pred_lgb_tuned, pos_label=1)],
    'Recall':    [recall_score(y_test, y_pred_lr_tuned,  pos_label=1),
                  recall_score(y_test, y_pred_dt_tuned,  pos_label=1),
                  recall_score(y_test, y_pred_rf_tuned,  pos_label=1),
                  recall_score(y_test, y_pred_xgb_tuned, pos_label=1),
                  recall_score(y_test_cb, y_pred_cb_tuned, pos_label=1),
                  recall_score(y_test, y_pred_lgb_tuned, pos_label=1)],
    'F1':        [f1_score(y_test, y_pred_lr_tuned,  pos_label=1),
                  f1_score(y_test, y_pred_dt_tuned,  pos_label=1),
                  f1_score(y_test, y_pred_rf_tuned,  pos_label=1),
                  f1_score(y_test, y_pred_xgb_tuned, pos_label=1),
                  f1_score(y_test_cb, y_pred_cb_tuned, pos_label=1),
                  f1_score(y_test, y_pred_lgb_tuned, pos_label=1)],
    'AUC-PR':    [pr_auc_lr, pr_auc_dt, pr_auc_rf, pr_auc_xgb,
                  pr_auc_cb, pr_auc_lgb],
    'AUC-ROC':   [roc_auc_score(y_test, best_lr.predict_proba(X_test)[:,1]),
                  roc_auc_score(y_test, best_dt.predict_proba(X_test)[:,1]),
                  roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1]),
                  roc_auc_score(y_test, y_prob_xgb),
                  roc_auc_score(y_test_cb, y_prob_cb),
                  roc_auc_score(y_test, y_prob_lgb)],
})

print("Final Results -- Training-selected thresholds (CatBoost: inner validation)")
print(results.round(4).to_string(index=False))
print()
print("Random baselines:  AUC-PR = 0.06 (positive rate)   AUC-ROC = 0.50")

### 3.2 Final Comparison Charts


In [ ]:
# Final comparison chart - all 6 models, AUC-PR and AUC-ROC side by side
# HJ: chart extended to 6 models + dual AUC metrics edited
models_plot = ['LR', 'DT', 'RF', 'XGB', 'CatBoost', 'LightGBM']
metrics_to_plot = ['Precision', 'Recall', 'F1', 'AUC-PR', 'AUC-ROC']

x = np.arange(len(models_plot))
width = 0.16
colors = ['#4C8BE0', '#7AB87A', '#E0A04C', '#E05C4C', '#9C7AE0']

fig, ax = plt.subplots(figsize=(13, 5))
for i, m in enumerate(metrics_to_plot):
    vals = results[m].values
    bars = ax.bar(x + (i - 2) * width, vals, width, label=m,
                  color=colors[i], edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{v:.2f}', ha='center', fontsize=7)

ax.axhline(0.06, color='gray', linestyle=':', linewidth=1,
           alpha=0.7, label='AUC-PR baseline (0.06)')
ax.axhline(0.50, color='gray', linestyle='--', linewidth=1,
           alpha=0.7, label='AUC-ROC baseline (0.50)')
ax.set_xticks(x)
ax.set_xticklabels(models_plot)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Cross-Model Performance: All Five Metrics for the Fraud Class',
             fontweight='bold')
ax.legend(loc='upper left', ncol=2, fontsize=9)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 3.3 Cross-Model Feature Importance Comparison


In [ ]:
top_n = 10
imp_dt_s  = pd.Series(best_dt.named_steps['dt'].feature_importances_,   index=X_train.columns)
imp_rf_s  = pd.Series(best_rf.named_steps['rf'].feature_importances_,   index=X_train.columns)
imp_xgb_s = pd.Series(best_xgb.named_steps['xgb'].feature_importances_, index=X_train.columns)
coef_lr_s = pd.Series(best_lr.named_steps['lr'].coef_[0],               index=X_train.columns).abs()

def norm(s):
    return s / s.max() if s.max() > 0 else s

imp_df = pd.DataFrame({
    'Decision Tree':  norm(imp_dt_s),
    'Random Forest':  norm(imp_rf_s),
    'XGBoost':        norm(imp_xgb_s),
    'LR Coefficient': norm(coef_lr_s),
}).fillna(0)

imp_df['Mean'] = imp_df.mean(axis=1)
top_features = imp_df.nlargest(top_n, 'Mean').drop(columns='Mean')

fig, ax = plt.subplots(figsize=(10, 6))
top_features.plot(kind='barh', ax=ax, width=0.7, alpha=0.85,
                  color=['#E05C4C', '#4CAF7D', '#7B6EE8', '#4C8BE0'])
ax.set_title(f'Top {top_n} Features — Normalized Importance Across All Models', fontweight='bold')
ax.set_xlabel('Normalized Importance')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

### 3.4 Analysis and Business Implications


## Interpretation limits

- Test results describe a single historical benchmark split, not deployment savings.
- The same dataset was explored during coursework; this is not a new untouched external test.
- Hyperparameters are selected on the training set and then reused for OOF thresholding; this is not fully nested cross-validation.
- Categorical vocabulary is encoded before the train/test split in the retained course workflow. A production pipeline should fit preprocessing within training folds.
- CatBoost selects its threshold on an inner validation split and refits; other models use OOF training predictions. Threshold transfer after refitting is a limitation.
- Cost assumptions, feature availability at claim submission, fairness and investigator capacity need independent validation.
- The unweighted/weighted diagnostic also changes tuned hyperparameters and thresholds; it does not isolate the causal effect of class weights.


In [ ]:
from pathlib import Path
import json
output_dir = ROOT / 'outputs'
output_dir.mkdir(exist_ok=True)
results.to_csv(output_dir / 'model_comparison.csv', index=False)
(output_dir / 'cost_comparison.json').write_text(json.dumps({
    'selection_data': 'out-of-fold training predictions',
    'cost_threshold': float(cost_thresh_xgb),
    'f1_threshold': float(best_thresh_xgb),
    'test_modeled_cost_f1': float(f1_test_cost),
    'test_modeled_cost_cost': float(cost_test_cost),
    'false_negative_cost_assumption': C_MISS,
    'false_positive_cost_assumption': C_FP,
}, indent=2))
print('Saved aggregate model metrics and assumed-cost comparison to outputs/')
